# Release assets

Collects the two files the release needs that the analysis notebooks do not
produce in the right place:

1. `vendor/gemma4_patched.py` — downloaded from the Gemma 4 NVFP4 model repo
2. `artifacts/sdg_direction_uncertain_for_review.csv` — copied from `sdggraph/`
   after `01_matrix_preprocessor.ipynb` has run

Run once. Nothing here touches the analysis notebooks, the caches, or any
result. Safe to re-run: existing files are left alone unless `OVERWRITE = True`.


In [ ]:
OVERWRITE = False   # set True to refetch/recopy over existing files

import os, sys, shutil, hashlib, json, datetime

RELEASE = os.environ.get('SDG_RELEASE_DIR')
if not RELEASE:
    if 'google.colab' in sys.modules or os.path.isdir('/content'):
        from google.colab import drive
        if not os.path.exists('/content/drive/MyDrive'):
            drive.mount('/content/drive')
        RELEASE = '/content/drive/MyDrive/sdg-llm-graph'
    else:
        RELEASE = os.path.abspath('..')

VENDOR    = os.path.join(RELEASE, 'vendor')
ARTIFACTS = os.path.join(RELEASE, 'artifacts')
GRAPH_DIR = os.path.join(RELEASE, 'sdggraph')
os.makedirs(VENDOR, exist_ok=True)
os.makedirs(ARTIFACTS, exist_ok=True)

print('release dir :', RELEASE)
if not os.path.exists(os.path.join(RELEASE, 'README.md')):
    print('WARNING: no README.md here — is this the release folder?')


## 1. Gemma 4 NVFP4 vLLM patch

`gemma4_patched.py` replaces vLLM's `model_executor/models/gemma4.py`. Without
it, NVFP4 scale keys (`.weight_scale`, `.weight_scale_2`, `.input_scale`) fail
to map onto FusedMoE parameter names, so the quantised checkpoint will not load
(vLLM issue #38912). Needed by the `gemma4-26b` backbone only.

The file is Apache-2.0, derived from vLLM's own Gemma 4 implementation, so it
can be redistributed with attribution — which the generated `NOTICE` provides.


In [ ]:
PATCH_URL = ('https://huggingface.co/bg-digitalservices/Gemma-4-26B-A4B-it-NVFP4/'
             'resolve/main/gemma4_patched.py')
dest = os.path.join(VENDOR, 'gemma4_patched.py')

if os.path.exists(dest) and not OVERWRITE:
    text = open(dest).read()
    print(f'already present ({len(text):,} chars) — set OVERWRITE=True to refetch')
else:
    import requests
    r = requests.get(PATCH_URL, timeout=60)
    r.raise_for_status()
    if len(r.text) < 1000 or 'gemma' not in r.text.lower():
        raise RuntimeError('downloaded file does not look like the patch; '
                           'fetch it by hand from the model repo Files tab')
    text = r.text
    open(dest, 'w').write(text)
    print(f'downloaded {len(text):,} chars -> {dest}')

PATCH_SHA = hashlib.sha256(text.encode()).hexdigest()
print('sha256:', PATCH_SHA)
print('lines :', text.count(chr(10)) + 1)


## 2. Direction-of-progress exclusion list

The 216 UN indicator series whose direction of progress the keyword classifier
could not assign confidently, and which were therefore excluded rather than
defaulted. Written by `01_matrix_preprocessor.ipynb`; this copies it next to
the matrix it belongs to.


In [ ]:
src = os.path.join(GRAPH_DIR, 'sdg_direction_uncertain_for_review.csv')
dst = os.path.join(ARTIFACTS, 'sdg_direction_uncertain_for_review.csv')

if not os.path.exists(src):
    print(f'NOT FOUND: {src}')
    print('Run 01_matrix_preprocessor.ipynb first — it writes this file.')
elif os.path.exists(dst) and not OVERWRITE:
    print('already in artifacts/ — set OVERWRITE=True to replace')
else:
    shutil.copy(src, dst)
    print('copied ->', dst)

if os.path.exists(dst):
    rows = open(dst).read().strip().split(chr(10))
    print(f'{len(rows) - 1} series (expected 216)')
    for line in rows[:4]:
        print('   ', line[:100])


## 3. NOTICE file and README values

In [ ]:
notice = f'''sdg-llm-graph
Copyright (c) 2026 the authors of "Neighbours and Graphs: LLM-Based SDG
Classification of Research Abstracts"

Licensed under the Apache License, Version 2.0.

This distribution includes vendor/gemma4_patched.py, retrieved from the
Hugging Face model repository bg-digitalservices/Gemma-4-26B-A4B-it-NVFP4
(Mario Iseli). That file is derived from the Apache-2.0 licensed Gemma 4
implementation in vLLM (https://github.com/vllm-project/vllm) and is
redistributed here under the same licence.

  retrieved : {datetime.date.today().isoformat()}
  source    : {PATCH_URL}
  sha256    : {PATCH_SHA}
'''
open(os.path.join(RELEASE, 'NOTICE'), 'w').write(notice)
print(notice)

zp = os.path.join(GRAPH_DIR, 'SDG_UN_data.zip')
if os.path.exists(zp):
    h = hashlib.sha256()
    with open(zp, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    print(f'SDG_UN_data.zip  {os.path.getsize(zp):,} B')
    print(f'  sha256: {h.hexdigest()}')
    print('  -> paste into the README provenance table (release 2026.Q1.G.01)')
else:
    print('SDG_UN_data.zip not in sdggraph/ — skipping its checksum')


In [ ]:
print('Release folder contents:')
for d in ['', 'notebooks', 'artifacts', 'vendor', 'scripts']:
    p = os.path.join(RELEASE, d)
    if not os.path.isdir(p):
        continue
    print(f'\n  {d or "."}/')
    for f in sorted(os.listdir(p)):
        fp = os.path.join(p, f)
        if os.path.isfile(fp):
            print(f'    {f:44s} {os.path.getsize(fp):>10,} B')

print('\nStill to do by hand: full Apache-2.0 text in LICENSE, <repo-url> '
      'placeholders, Aurora versioned DOI, OpenAlex snapshot date.')
print('Do NOT ship any notebooks/*_VERIFY_*.ipynb file.')
